In [0]:
xml_path = "/Volumes/workspace/default/sushreevolume/cleaned_cost_submission.xml"

print(xml_path)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/sushreevolume/"))

In [0]:
print(dbutils.fs.head(xml_path, 1000))

In [0]:
header_df = (
    spark.read
    .format("xml")
    .option("rowTag", "Header")
    .option("inferSchema", "true")
    .load(xml_path)
)

display(header_df)

In [0]:
claims_raw_df = (
    spark.read
    .format("xml")
    .option("rowTag", "Claim")
    .option("inferSchema", "true")
    .load(xml_path)
)

claims_raw_df.printSchema()

In [0]:
claims_raw_df.display()

In [0]:
from pyspark.sql.functions import col

claims_flat_df = claims_raw_df.select(
    col("ID").alias("claim_id"),
    col("ProviderID").alias("provider_id"),

    col("Encounter.FacilityID").alias("facility_id"),
    col("Encounter.ID").alias("encounter_id"),
    col("Encounter.PatientID").alias("patient_id"),
    col("Encounter.Type").alias("encounter_type"),
    col("Encounter.Start").alias("encounter_start"),
    col("Encounter.End").alias("encounter_end"),
    col("Encounter.StartType").alias("start_type"),
    col("Encounter.Specialty").alias("specialty"),
    col("Encounter.SubSpecialty").alias("sub_specialty"),
    col("Encounter.DRGCode").alias("drg_code"),
    col("Encounter.EndType").alias("end_type"),

    col("Encounter.Diagnosis.Type").alias("diagnosis_type"),
    col("Encounter.Diagnosis.Code").alias("diagnosis_code"),

    col("Encounter.Procedure.Type").alias("procedure_type"),
    col("Encounter.Procedure.Code").alias("procedure_code"),
    col("Encounter.Procedure.Duration").alias("procedure_duration"),

    col("Encounter.EDTriageLevel").alias("ed_triage_level"),
    col("Encounter.OPAttendanceType").alias("op_attendance_type"),
    col("Encounter.TheatreAttendanceFlag").alias("theatre_attendance_flag"),
    col("Encounter.CriticalCareTime").alias("critical_care_time"),
    col("Encounter.VentilationTime").alias("ventilation_time"),

    # CostBucketDirect
    col("Encounter.CostBucketDirect.Allied").alias("direct_allied"),
    col("Encounter.CostBucketDirect.Anaesthesia").alias("direct_anaesthesia"),
    col("Encounter.CostBucketDirect.ED").alias("direct_ed"),
    col("Encounter.CostBucketDirect.ICU").alias("direct_icu"),
    col("Encounter.CostBucketDirect.Imaging").alias("direct_imaging"),
    col("Encounter.CostBucketDirect.Laboratory").alias("direct_laboratory"),
    col("Encounter.CostBucketDirect.Physician").alias("direct_physician"),
    col("Encounter.CostBucketDirect.OP").alias("direct_op"),
    col("Encounter.CostBucketDirect.OR").alias("direct_or"),
    col("Encounter.CostBucketDirect.Other").alias("direct_other"),
    col("Encounter.CostBucketDirect.Pharmacy").alias("direct_pharmacy"),
    col("Encounter.CostBucketDirect.Prosthesis").alias("direct_prosthesis"),
    col("Encounter.CostBucketDirect.Supplies").alias("direct_supplies"),
    col("Encounter.CostBucketDirect.SPS").alias("direct_sps"),
    col("Encounter.CostBucketDirect.Ward").alias("direct_ward"),

    # CostBucketOverheads
    col("Encounter.CostBucketOverheads.Allied").alias("overhead_allied"),
    col("Encounter.CostBucketOverheads.Anaesthesia").alias("overhead_anaesthesia"),
    col("Encounter.CostBucketOverheads.ED").alias("overhead_ed"),
    col("Encounter.CostBucketOverheads.ICU").alias("overhead_icu"),
    col("Encounter.CostBucketOverheads.Imaging").alias("overhead_imaging"),
    col("Encounter.CostBucketOverheads.Laboratory").alias("overhead_laboratory"),
    col("Encounter.CostBucketOverheads.Physician").alias("overhead_physician"),
    col("Encounter.CostBucketOverheads.OP").alias("overhead_op"),
    col("Encounter.CostBucketOverheads.OR").alias("overhead_or"),
    col("Encounter.CostBucketOverheads.Other").alias("overhead_other"),
    col("Encounter.CostBucketOverheads.Pharmacy").alias("overhead_pharmacy"),
    col("Encounter.CostBucketOverheads.Prosthesis").alias("overhead_prosthesis"),
    col("Encounter.CostBucketOverheads.Supplies").alias("overhead_supplies"),
    col("Encounter.CostBucketOverheads.SPS").alias("overhead_sps"),
    col("Encounter.CostBucketOverheads.Ward").alias("overhead_ward"),

    # CostTypeDetails
    col("Encounter.CostTypeDetails.SWNurs").alias("cost_sw_nurs"),
    col("Encounter.CostTypeDetails.SWDoc").alias("cost_sw_doc"),
    col("Encounter.CostTypeDetails.SWAllied").alias("cost_sw_allied"),
    col("Encounter.CostTypeDetails.SWNonClin").alias("cost_sw_non_clin"),
    col("Encounter.CostTypeDetails.Laboratory").alias("cost_laboratory"),
    col("Encounter.CostTypeDetails.Imaging").alias("cost_imaging"),
    col("Encounter.CostTypeDetails.Pharmacy").alias("cost_pharmacy"),
    col("Encounter.CostTypeDetails.Prostheses").alias("cost_prostheses"),
    col("Encounter.CostTypeDetails.MS").alias("cost_ms"),
    col("Encounter.CostTypeDetails.Hotel").alias("cost_hotel"),
    col("Encounter.CostTypeDetails.GS").alias("cost_gs"),
    col("Encounter.CostTypeDetails.Depreciation").alias("cost_depreciation"),
    col("Encounter.CostTypeDetails.OHF").alias("cost_ohf"),
    col("Encounter.CostTypeDetails.OHC").alias("cost_ohc"),
    col("Encounter.CostTypeDetails.NABMs").alias("cost_nabms")
)

display(claims_flat_df)

In [0]:
claim_count = claims_flat_df.count()
print("Claim row count:", claim_count)

In [0]:
from pyspark.sql.functions import lit

header_row = header_df.first()

claims_final_df = (
    claims_flat_df
    .withColumn("sender_id", lit(header_row["SenderID"]))
    .withColumn("receiver_id", lit(header_row["ReceiverID"]))
    .withColumn("transaction_date", lit(header_row["TransactionDate"]))
    .withColumn("record_count_from_header", lit(header_row["RecordCount"]))
    .withColumn("disposition_flag", lit(header_row["DispositionFlag"]))
)

display(claims_final_df)

In [0]:
from pyspark.sql.functions import to_timestamp

claims_final_df = (
    claims_final_df
    .withColumn("encounter_start_ts", to_timestamp(col("encounter_start"), "dd/MM/yyyy HH:mm"))
    .withColumn("encounter_end_ts", to_timestamp(col("encounter_end"), "dd/MM/yyyy HH:mm"))
    .withColumn("transaction_ts", to_timestamp(col("transaction_date"), "dd/MM/yyyy HH:mm"))
)

display(claims_final_df)

In [0]:
target_table = "workspace.default.cost_submission_claims"

(
    claims_final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"Table created successfully: {target_table}")

In [0]:
%sql
SELECT *
FROM workspace.default.cost_submission_claims
LIMIT 20